# 00 · Questions, outcomes and data

This notebook states the six questions *before* any result. Notebooks 01–03 answer them in a fixed
order — **easy synthetic → harder synthetic → real data** — and each section ends with an
*Observed outcome* line chosen by a rule written in code here, not by reading the plot.

**The setting in one line.** One set of factor scores $Z$ ($N\times K$) drives every view:
$Y^{(m)}\approx Z\,W^{(m)\top}$. Factor $k$ is **shared** if its loading column $W^{(m)}_{\cdot k}$ is non-zero in
every view, **private** to view $m$ if only there. The *pattern* of factor $k$ is the 0/1 vector
$H_{\cdot k}$ over views. "Shared" in these notebooks always means *shared between views*, never
between cohorts.

**Development note (honesty).** Before these notebooks were written, a pilot on easy data showed
that (a) a learned cohort $\pi$ drifts towards 1 and (b) a cold-started view switch stays on.
Both observations shaped two *extra* arms — the paired cohort factorisation (Q1) and the warm-started
switch (Q4) — but the original arms are kept and the outcomes below were fixed before the runs
reported in notebooks 01–03.

In [ ]:
import os, sys, warnings
os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks_sharedness" else os.getcwd()
sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display
from sharedness import experiments as ex, metrics as mt
from sharedness.data import make_view_data, H_EASY, pattern_for

pd.set_option("display.precision", 3)
sns.set_theme(style="whitegrid", context="notebook")
RESULTS = os.path.join(ROOT, "notebooks_sharedness", "results")
os.makedirs(RESULTS, exist_ok=True)

def outcome(code_text, title="Observed outcome"):
    code, text = code_text
    display(Markdown(f"> **{title}: {code}** — {text}"))
    return code

## Q1 — Can CohortFACTM learn how often cohorts differ?

**Why we ask.** The cohort switch rate $\pi$ is fixed at 0.5 in CohortFACTM. If it can be learned, the model
reports how many cohorts actually have their own mean; if the learned value is biased, we need to know why
before trusting any cohort read-out.

**Setup.**

- **Samples.** $C=6$ cohorts, $n_c=50$ samples each, $N=300$. (This is $C$ larger and $n_c$ smaller than the
  other questions, because Q1 is about cohorts, not views.)
- **Factors.** $K$ as in this level. Each true factor score is $z_{nk}=\gamma^\ast_{ck}\delta^\ast_{ck}+\varepsilon$:
  a cohort either sits at the common centre or is shifted by $\delta^\ast=\pm2$.
- **The knob.** The true switch rate $\pi^\ast$ — the fraction of $(c,k)$ pairs that are shifted — is set to
  **0.1, 0.3, 0.6** in turn. These are the numbers the learned $\mathbb E[\pi]$ should recover.
- **What counts as a real shift.** The data are centred on the grand mean, so a switch can only see a cohort's
  mean *relative to the grand mean*. The truth we score against is therefore
  $G^{\text{eff}}_{ck}=\mathbb 1\{|\bar z_{ck}-\bar z_{\cdot k}|>1\}$, not the raw $\gamma^\ast$. $G^{\text{eff}}$ can differ
  from $\pi^\ast$ (e.g. when one cohort shifts, the others move a little the other way).
- **Repeats.** 5 seeds per rate.

**What we fit.** Three cohort priors, all with ARD loadings:
1. **$\pi=0.5$ fixed** — CohortFACTM as it is today.
2. **$\pi$ learned, current factorisation** $q(\gamma)q(\delta)$ — $q(\pi)=\mathrm{Beta}(1+\sum_{c,k}\mathbb E\gamma_{ck},\,1+CK-\sum\mathbb E\gamma_{ck})$.
3. **$\pi$ learned, paired factorisation** $q(\gamma)q(\delta\mid\gamma)$ — the fix proposed in the model document.

**What we measure.**

- **$\mathbb E[\pi\mid Y]$** vs the true rate (effective rate on synthetic data). Good = close to the diagonal.
- **$\gamma$ AUC** $=P(\mathbb E\gamma_{ck}>\mathbb E\gamma_{c'k'}\mid G^{\text{eff}}_{ck}=1,\,G^{\text{eff}}_{c'k'}=0)$: does
  $\mathbb E\gamma$ rank the real shifts above the non-shifts? $1$ perfect, $0.5$ chance.
- **null $\mathbb E\gamma$** = mean $\mathbb E\gamma$ where the truth is 0 (should be low); **alt** = where it is 1.

**How to read the answer** (the rule the code applies, fixed before the run):

| Outcome | If we see | It would mean | So we |
|---|---|---|---|
| **O1** Learnable | max $|\mathbb E\pi-\pi^\ast|<0.15$ and $\mathbb E\pi$ rises with $\pi^\ast$ (slope $>0.5$) | the data carry the rate | keep $\pi$ learnable |
| **O2** Drifts to 1 | $\mathbb E\pi>0.8$ whatever $\pi^\ast$ is | switches that should be off are not pushed off | the factorisation, not the formula, is at fault — try the paired one |
| **O3** Prior-dominated | $\mathbb E\pi$ moves $<0.2$ across all $\pi^\ast$ | too few switches ($CK$) to learn from | leave $\pi$ fixed |
| **O4** Partial | anything else (moves with the truth but biased) | the rate is partly recoverable | report the bias and prefer the arm with the smallest one |

## Q2 — Does a plain fit already show which factors are shared?

**Why we ask.** If an ordinary fit already reveals the shared/private pattern, no new model is needed to *see* it —
only (later) to *identify* it. This tells us whether the hard part is reading the pattern or finding the factors.

**Setup.**

- **Samples / views / noise / loadings** exactly as this level's data (stated at the top of the notebook: easy
  = 4 views $\times$ 30 features, $N=300$, noise 0.5, independent loadings; hard = 4 views of 80/40/15/10
  features, $N=150$, noise 1.2, correlated loadings, one weak factor, 2 surplus fitted factors).
- **True pattern $H$** (views $\times$ factors) is the matrix drawn at the top of notebook 00. We know it, so we
  can score recovery.

**What we fit.** Two loadings priors: **no prior** ($w\sim\mathcal N(0,1)$) and **ARD**. No cohort structure.

**What we measure.**

- **Variance share** $r_{mk}=\lVert\mathbb E z_{\cdot k}\,\mathbb E w^{(m)\top}_{\cdot k}\rVert_F^2/\lVert Y^{(m)}\rVert_F^2$:
  the fraction of view $m$'s variance carried by factor $k$. Normalise per factor to activity $a_{mk}\in[0,1]$.
- **Pattern AUC** $=P(a_{mk}>a_{m'k'}\mid H_{mk}=1,\,H_{m'k'}=0)$, after matching fitted to true factors
  (Hungarian on $|\mathrm{corr}|$). Threshold-free; $1$ perfect, $0.5$ chance. We report it with and without ARD.

**How to read the answer** (the rule the code applies, fixed before the run):

| Outcome | If we see | It would mean | So we |
|---|---|---|---|
| **O1** Yes, any fit | AUC $\ge0.95$ with and without ARD | sharedness is a read-out of the fit | no new model needed to see the pattern |
| **O2** Yes, with ARD | AUC $\ge0.95$ only with ARD | ARD is what makes the pattern readable | use ARD loadings when reading sharedness |
| **O3** Partly | best AUC in $[0.75,0.95)$ | some cells are read wrongly | inspect leaks (private read as shared) vs misses |
| **O4** No | best AUC $<0.75$ | the fit does not expose the pattern | the bottleneck is elsewhere (see Q2b) |

## Q2b — When the fit lands on the wrong factors, is it the optimiser or the model?

**Why we ask.** If the model's objective (ELBO) *prefers* the true factors but the algorithm fails to reach them, the
fix is better optimisation (cheap). If the ELBO itself prefers a rotated answer, the fix is new theory (hard).
Everything downstream depends on which it is.

**Setup.**

- **Same data as Q2.** Look only at seeds where the ordinary ARD fit (from the usual PCA start) missed the true
  factors: $\min_k|\mathrm{corr}(\hat z_k,z^\ast_k)|<0.9$.
- For each such seed, **refit a second time starting exactly at the truth** $(Z^\ast,W^\ast)$, and compare the two
  fits' final ELBO.

**What we fit.** ARD loadings from two starts: the usual PCA start, and the true $(Z^\ast,W^\ast)$.

**What we measure.**

- **$\Delta_{\text{start}}=\mathrm{ELBO}_{\text{truth start}}-\mathrm{ELBO}_{\text{PCA start}}$** (nats). Positive means the
  model scores the truth higher than the rotated answer it actually found.
- Whether the truth-started fit **keeps** the true axes (does not drift back to the rotated answer).

**How to read the answer** (the rule the code applies, fixed before the run):

| Outcome | If we see | It would mean | So we |
|---|---|---|---|
| **O0** Not needed | every ARD fit already found the true axes | no problem to diagnose | move on |
| **O1** Optimiser | $\Delta_{\text{start}}>10$ nats and the truth start keeps the true axes | the ELBO prefers the truth; the algorithm just doesn't reach it | better initialisation / multi-start fixes it |
| **O2** Model | $\Delta_{\text{start}}<-10$, or the truth start drifts away | the ELBO itself prefers a rotation | an identification result is needed (mean-field artefact regime) |
| **O3** Flat | $|\Delta_{\text{start}}|\le10$ | the two solutions are indistinguishable to the ELBO | the rotation is (nearly) unidentified |

## Q3 — How many views are needed before shared-vs-private is a meaningful question?

**Why we ask.** With 2 views there are only 3 possible patterns; with 4 there are 15. We need to know whether few
views make the pattern merely *coarse* or actually *hard to detect* — this decides whether COVID ($M=2$) is a
usable test bed.

**Setup.**

- **A ladder of view counts** $M\in\{2,3,4,6\}$. For each $M$ the true pattern has one fully shared factor, one
  factor in views 1–2 (when $M>2$), and one factor private to each view.
- Feature counts per view: 30 (easy) or spread 80…10 (hard); other settings as this level.
- **Repeats.** 5 seeds per $M$.
- Plus a **no-fitting table**: the posterior sd of a per-factor rate learned from only $M$ switches,
  $\mathrm{Beta}(1+s_k,1+M-s_k)$ — shows how uncertain $\pi_k$ must be when $M$ is small, before any experiment.

**What we fit.** ARD loadings, no cohort structure, one fit per $(M,\text{seed})$.

**What we measure.** **AUC of ARD activity against $H$** (as in Q2), averaged over seeds, for each $M$. Good = high and
flat across $M$; a rise with $M$ means more views help detection.

**How to read the answer** (the rule the code applies, fixed before the run):

| Outcome | If we see | It would mean | So we |
|---|---|---|---|
| **O1** M does not limit detection | AUC $\ge0.95$ at every $M$ | even $M=2$ detects the pattern, just coarsely | COVID is usable; more views only add resolution |
| **O2** More views help | AUC rises by $\ge0.05$ from the smallest to the largest $M$ | detection improves with views | prefer the four-view data over COVID |
| **O3** More views hurt | AUC falls by $\ge0.05$ | more private factors split the signal | cap the number of views / factors |
| **O4** Flat but imperfect | otherwise | detection is limited by something other than $M$ | look at noise / factor strength instead |

## Q4 — Does an explicit on/off view switch beat ARD's soft dials?

**Why we ask.** ARD turns a factor's loadings in a view *down* but never fully *off*, so any yes/no call needs an
arbitrary cutoff. A real switch $h_{mk}$ would give a direct answer — if it is identifiable.

**Setup.**

- **Same data as Q2** (this level). We know $H$, so we can check the switch's calls.
- The switch is $w^{(m)}_{\cdot k}=h_{mk}V^{(m)}_{\cdot k}$, $h_{mk}\sim\mathrm{Bern}(\pi)$, one Bernoulli per (view, factor).

**What we fit.** Four arms: **ARD** (reference, no switch); **switch, cold start**; **switch, warm start** (started from a
converged ARD fit); **switch, warm start + fixed slab** (slab precision $\alpha\equiv1$ instead of learned ARD).

**What we measure.**

- **Switch AUC** of $\mathbb E[h]$ against $H$ (as in Q2).
- **Decisiveness** = share of $\mathbb E[h_{mk}]$ outside $(0.1,0.9)$ — are the calls confident?
- **Switches on** = share with $\mathbb E[h]>0.5$ (true value 50% in the synthetic pattern).
- **Leaks** (private called shared) and **misses** (shared called absent).

**How to read the answer** (the rule the code applies, fixed before the run):

| Outcome | If we see | It would mean | So we |
|---|---|---|---|
| **O1** Cleaner | best switch AUC $\ge$ ARD$-0.02$, $\ge0.9$, and $\ge90\%$ decisive | the switch gives sharper correct calls | worth adding as a diagnostic |
| **O2** Switch stays on | $\ge95\%$ of switches on in the best arm | $h$ cannot tell itself apart from a shrunk slab ($\alpha\to\infty$ and $h=0$ look the same) | drop the switch or use a fixed slab |
| **O3** Worse than ARD | best switch AUC $<$ ARD$-0.05$ | the switch harms recovery | keep ARD |
| **O4** Mixed | otherwise | it depends on the slab | report per arm |

## Q5 — Do forced zeros pin the factors down better than learned zeros?

**Why we ask.** This is the headline. A *frozen* zero (‘factor 2 never appears in view 1’) should make a
shared↔private rotation impossible; a *learned* zero can be switched back on. If the frozen penalty grows with
$N$ and the learned one does not, only a prescribed pattern identifies the split.

**Setup.**

- **The rotation.** Pick the fully shared factor and one view-private factor. Rotate one into the other by an
  angle $\theta\in\{0,10,20,30,45\}^\circ$: set the factor means to $Z^\ast G(\theta)$.
- **Freeze those means** and re-optimise *everything else* (loadings, switches, noise, cohort parameters).
- **Two sample sizes.** $N=150$ ($n_c=50$) and $N=600$ ($n_c=200$) — a $\times4$ increase, to see whether the
  penalty scales with $N$. With $z\sim\mathcal N(0,I)$ the factor-prior part of the ELBO is unchanged by the
  rotation, so any gap comes from the loadings side.
- **Repeats.** 2 seeds (the signal is large).

**What we fit.** Three arms, each re-optimised at every angle: **ARD** (no view structure); **learned switch**;
**frozen pattern** (the true $H$ on synthetic data; the read-off pattern on real data).

**What we measure.** **$\Delta(\theta)=\mathrm{ELBO}(0)-\mathrm{ELBO}(\theta)$** (nats): how hard the model resists the rotation.
We read its **growth from $N$ to $4N$**: a ratio near $4$ means the penalty scales with $N$ (likelihood-level
identification); a ratio near $1$ means a fixed, prior-level preference.

**How to read the answer** (the rule the code applies, fixed before the run):

| Outcome | If we see | It would mean | So we |
|---|---|---|---|
| **O1** Flat | $\Delta<5$ nats at the larger $N$ | rotating shared into private is essentially free | no identification from this mechanism |
| **O2** Grows like N | gap ratio $\ge0.6\times$ the sample-size ratio | the split is pinned at the likelihood level | a prescribed pattern identifies it (the `lem:zeros` regime) |
| **O3** Grows slower than N | otherwise | there is a real penalty but it does not scale with $N$ | a preference, not identification |

## Q6 — One sharing rate for the whole model, or one per factor?

**Why we ask.** A single $\pi$ answers ‘are this dataset's factors mostly shared?’; a per-factor $\pi_k$ answers ‘is
factor $k$ shared?’ — but $\pi_k$ is learned from only $M$ switches. We need to know if the finer version is worth
the extra uncertainty.

**Setup.**

- **View counts** $M\in\{2,4,8\}$, same pattern family as Q3, this level's settings.
- Fit the view switch twice: with **one global $\pi$** and with **one $\pi_k$ per factor**.
- **Repeats.** 5 seeds per $M$.

**What we fit.** View switch (fixed slab), global $q(\pi)=\mathrm{Beta}(1+\sum_{m,k}\mathbb E h_{mk},\,1+MK-\sum\mathbb E h_{mk})$
vs per-factor $q(\pi_k)=\mathrm{Beta}(1+\sum_m\mathbb E h_{mk},\,1+M-\sum_m\mathbb E h_{mk})$.

**What we measure.**

- **Switch AUC** in both modes (does per-factor recover the pattern any better?).
- **$\mathrm{sd}(\pi_k)$** — how uncertain the per-factor rate is (large when $M$ is small).
- **Separation** = mean $\mathbb E[\pi_k]$ over shared factors minus over private factors.

**How to read the answer** (the rule the code applies, fixed before the run):

| Outcome | If we see | It would mean | So we |
|---|---|---|---|
| **O1** Either works | $\mathrm{sd}(\pi_k)<0.15$ at $M\le4$ and no AUC gain | per-factor is certain enough and costs nothing | choose by interpretability |
| **O2** Use one global $\pi$ | $\mathrm{sd}(\pi_k)\ge0.15$ at $M\le4$ and no AUC gain | per-factor rates are noise at realistic $M$ | use a single $\pi$ |
| **O3** Per-factor helps | switch AUC gain $\ge0.03$ at some $M$ | the finer rate improves recovery there | keep $\pi_k$ where it helps |

## The difficulty ladder

| | Easy synthetic | Harder synthetic | Real data |
|---|---|---|---|
| views $M$ | 4 × 30 features | 4 views, 80 / 40 / 15 / 10 features (unbalanced) | COVID: 2 · four-view cohort table: 4 |
| samples $N$ | 300 (3 cohorts) | 150 (3 cohorts) | 378 · 1038 |
| noise sd | 0.5 | 1.2 | unknown |
| loadings | independent | correlated, $\rho=0.5$ | unknown |
| factors | 4, all strong | 6, one weak (scale 0.4) | unknown |
| $K$ fitted | = true | true + 2 (surplus factors) | 5 / 6 |
| truth | known | known | none → stability and identification only |

Why these knobs: correlated loadings make the mean-field-preferred axes differ from the truth; unbalanced
views let the big view dominate; a weak private factor is the hardest cell to call; surplus factors test
whether the fit invents patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (name, H) in zip(axes, [("easy", ex.LEVELS["easy"]["H"]), ("hard", ex.LEVELS["hard"]["H"])]):
    sns.heatmap(H, ax=ax, cmap="Blues", cbar=False, annot=True, linewidths=.5,
                xticklabels=[f"f{k}" for k in range(H.shape[1])],
                yticklabels=[f"view {m+1}" for m in range(H.shape[0])])
    ax.set_title(f"True pattern H — {name}  (1 = factor used by view)")
plt.tight_layout(); plt.show()

### Q3, the part that needs no fitting

How sure can a per-factor rate be, even with a perfectly observed pattern?

In [ ]:
tab = ex.q3_counting_table()
tab.to_csv(os.path.join(RESULTS, "q3_counting.csv"), index=False)
tab

Reading the table: at $M=2$ a shared factor's $\pi_k$ has posterior mean $0.75$ with sd $0.19$ and a private one
$0.50$ with sd $0.22$ — the two overlap almost completely. Per-factor rates only separate around $M\ge6$.

## Data inventory (what the real-data notebook can and cannot ask)

In [ ]:
from sharedness.data import load_covid_views, load_four_view_cohort
cov = load_covid_views(top=150, cohort_mode="grouped")
fv = load_four_view_cohort(top=100)
inv = pd.DataFrame([
    dict(dataset="COVID", M=cov.M, N=cov.N, cohorts=len(np.unique(cov.labels)),
         features=[Y.shape[1] for Y in cov.Ys], views=cov.view_names, patterns=2**cov.M - 1),
    dict(dataset="four-view cohort table", M=fv.M, N=fv.N, cohorts=len(np.unique(fv.labels)),
         features=[Y.shape[1] for Y in fv.Ys], views=fv.view_names, patterns=2**fv.M - 1),
])
inv

**Caveats carried into notebook 03.** The four-view table is the IMMUCAN-style long table in the repo
root, restricted to the 1038 samples that have all four views (6 cohorts; SCCHN3 has no complete sample).
DNA is binary mutation calls (≈99.6 % zeros): only mutations present in ≥1 % of samples are kept (25 mutations; at 5 % only 3 would survive), and
they are fitted with a Gaussian likelihood, which is an approximation. The IMMUCAN loader in
`immucan_data/` needs a Mac drive path and is not used.